# 05 — DistilBERT Classifier (Medical Abstracts)

**Task:** classify each medical abstract into one of 5 disease categories.
**Model:** `distilbert-base-uncased`, fine-tuned end to end (recommended in the Week 3 slides).

Rules followed (same as `04_baseline_benchmark`):
- Same `train.csv` / `validation.csv` / `test.csv` files (seed 42 split).
- Same `evaluate_model` function and **Macro F1** as the primary metric.
- All choices (epochs, learning rate, early stopping) are made on **validation only**.
- The **test set is used once**, in the final section, after all choices are fixed.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## 1. Setup

In [ ]:
!pip install -q transformers accelerate

import os, random, time
import numpy as np
import pandas as pd
import torch
import transformers

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU found. Training will be very slow. Switch the runtime to T4 GPU.")

## 2. Experiment configuration
Everything that defines this run is in one place, so it can be reported exactly.
To run the BERT comparison later, change only `MODEL_NAME` to `bert-base-uncased`.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"   # later: "bert-base-uncased"
MODEL_LABEL = "DistilBERT" if "distil" in MODEL_NAME else "BERT-base"

MAX_LENGTH = 512        # BERT's maximum input length (in tokens)
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 4          # upper limit; early stopping can stop earlier
WEIGHT_DECAY = 0.01     # L2-style regularization
WARMUP_RATIO = 0.1      # first 10% of steps: learning rate warms up from 0
EARLY_STOPPING_PATIENCE = 2

OUTPUT_DIR = f"outputs_{MODEL_LABEL.lower()}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Load the shared split

In [ ]:
# Upload train.csv, validation.csv and test.csv only if they are not already here.
needed = ["train.csv", "validation.csv", "test.csv"]
if not all(os.path.exists(f) for f in needed):
    from google.colab import files
    uploaded = files.upload()

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# Sanity checks: the split must be exactly the shared one.
assert len(train_df) == 5808 and len(val_df) == 1245 and len(test_df) == 1245
assert not set(train_df.abstract_id) & set(val_df.abstract_id)
assert not set(train_df.abstract_id) & set(test_df.abstract_id)
assert not set(val_df.abstract_id) & set(test_df.abstract_id)
print("Split checks passed: correct sizes, no overlap.")

display(train_df.head())

## 4. Labels and the shared evaluation function
The CSV labels are 1–5; PyTorch needs 0–4. We convert for training and convert back
before evaluation, so the numbers are directly comparable with the baseline notebook.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

LABEL_IDS = [1, 2, 3, 4, 5]
LABEL_NAMES = ["Neoplasms", "Digestive", "Nervous", "Cardiovascular", "General"]
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

def to_model_label(csv_label):   # 1..5 -> 0..4
    return int(csv_label) - 1

def to_csv_label(model_label):   # 0..4 -> 1..5
    return int(model_label) + 1

PRIMARY_METRIC = "Macro F1"

# Same function as 04_baseline_benchmark, so all models are measured identically.
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

print("Primary metric:", PRIMARY_METRIC)

## 5. Tokenizer and token-length check
BERT's limit is 512 **tokens**, not words. WordPiece splits rare medical terms into
several pieces, so we measure length with the real tokenizer (train set only).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "Hypertrophic cardiomyopathy with ventricular tachycardia"
print("Example:", tokenizer.tokenize(example))

train_token_lengths = np.array([
    len(ids) for ids in tokenizer(train_df["medical_abstract"].tolist(), truncation=False)["input_ids"]
])

print("\nTRAINING ABSTRACT LENGTH (tokens, incl. [CLS] and [SEP])")
print(f"Mean: {train_token_lengths.mean():.1f} | Median: {np.median(train_token_lengths):.0f} "
      f"| 95th pct: {np.percentile(train_token_lengths, 95):.0f} | Max: {train_token_lengths.max()}")
for limit in [128, 256, 384, 512]:
    pct = (train_token_lengths > limit).mean() * 100
    print(f"> {limit} tokens: {pct:.2f}% of abstracts would be truncated")

## 6. Tokenize the three splits

In [ ]:
from transformers import DataCollatorWithPadding

class AbstractDataset(torch.utils.data.Dataset):
    """Tokenized abstracts + 0-based labels. Padding is done per batch by the collator."""
    def __init__(self, df):
        self.encodings = tokenizer(
            df["medical_abstract"].tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.labels = [to_model_label(x) for x in df["condition_label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = AbstractDataset(train_df)
val_dataset = AbstractDataset(val_df)
test_dataset = AbstractDataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Datasets ready:", len(train_dataset), len(val_dataset), len(test_dataset))

## 7. Fine-tune
- Pre-trained DistilBERT + a new 5-class classification head.
- After every epoch the model is evaluated on **validation**; the epoch with the best
  Macro F1 is kept (`load_best_model_at_end`).
- **Early stopping**: if validation Macro F1 does not improve for 2 epochs, training stops.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
)

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

steps_per_epoch = int(np.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))
WARMUP_STEPS = int(WARMUP_RATIO * steps_per_epoch * NUM_EPOCHS)

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

start = time.time()
trainer.train()
TRAIN_TIME_SEC = time.time() - start
print(f"\nTraining time: {TRAIN_TIME_SEC / 60:.1f} minutes")

### Training curve
Training loss vs validation loss per epoch. If validation loss rises while training loss
keeps falling, the model is starting to overfit (that is what early stopping guards against).

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
train_loss = history.dropna(subset=["loss"])[["epoch", "loss"]]
val_hist = history.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_macro_f1"]]
display(val_hist.reset_index(drop=True))

plt.figure(figsize=(7, 4))
plt.plot(train_loss["epoch"], train_loss["loss"], marker="o", label="Train loss")
plt.plot(val_hist["epoch"], val_hist["eval_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(f"{MODEL_LABEL} - Training curve")
plt.legend(); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curve.png", dpi=150)
plt.show()

## 8. Validation results (same format as the baseline table)

In [ ]:
def predict(dataset):
    start = time.time()
    output = trainer.predict(dataset)
    seconds = time.time() - start
    probs = torch.softmax(torch.tensor(output.predictions), dim=-1).numpy()
    preds_csv = np.array([to_csv_label(p) for p in probs.argmax(axis=1)])
    return preds_csv, probs, seconds

val_preds, val_probs, val_time = predict(val_dataset)
y_val = val_df["condition_label"].values

val_results = evaluate_model(MODEL_LABEL, y_val, val_preds)
val_results.update({
    "Train Time (s)": round(TRAIN_TIME_SEC, 1),
    "Inference Time (s)": round(val_time, 2),
    "Checkpoint": MODEL_NAME,
    "Max Length": MAX_LENGTH,
    "Learning Rate": LEARNING_RATE,
    "Batch Size": TRAIN_BATCH_SIZE,
    "Best Epoch": int(val_hist.loc[val_hist["eval_macro_f1"].idxmax(), "epoch"]),
})

print("Validation metrics:")
for metric, value in val_results.items():
    print(f"{metric}: {value:.6f}" if isinstance(value, float) else f"{metric}: {value}")

print("\nClassification report:")
print(classification_report(y_val, val_preds, labels=LABEL_IDS, target_names=LABEL_NAMES, digits=4))

pd.DataFrame([val_results]).to_csv(f"{OUTPUT_DIR}/{MODEL_LABEL.lower()}_validation_results.csv", index=False)

### Comparison with the classical baselines (validation)
Upload `baseline_validation_results.csv` / `tuning_validation_results.csv` from the
baseline notebook if you want them in the same table.

In [ ]:
frames = [pd.DataFrame([val_results])]
for f in ["tuning_validation_results.csv", "baseline_validation_results.csv"]:
    if os.path.exists(f):
        frames.append(pd.read_csv(f))
        break

cols = ["Model", "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
comparison = (pd.concat(frames, ignore_index=True)[cols]
              .sort_values("Macro F1", ascending=False)
              .drop_duplicates("Model")          # best configuration per model
              .reset_index(drop=True))
display(comparison)

## 9. Validation confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_val, val_preds, labels=LABEL_IDS, display_labels=LABEL_NAMES,
    xticks_rotation=45, cmap="Blues"
)
plt.title(f"{MODEL_LABEL} - Validation Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/validation_confusion_matrix.png", dpi=150)
plt.show()

cm = pd.DataFrame(confusion_matrix(y_val, val_preds, labels=LABEL_IDS), index=LABEL_NAMES, columns=LABEL_NAMES)
print("Rows = actual labels; columns = predicted labels")
print(cm.to_string())

## 10. Error analysis (validation)
Which class pairs get confused most, and which mistakes the model made **with high confidence**.
Read 5–10 of these abstracts and write down why they might be confusing.

In [ ]:
pd.set_option("display.max_colwidth", 300)
name_of = dict(zip(LABEL_IDS, LABEL_NAMES))

val_analysis = val_df[["abstract_id", "medical_abstract"]].copy()
val_analysis["true"] = [name_of[x] for x in y_val]
val_analysis["predicted"] = [name_of[x] for x in val_preds]
val_analysis["confidence"] = val_probs.max(axis=1).round(3)
errors = val_analysis[val_analysis["true"] != val_analysis["predicted"]]

print(f"Misclassified: {len(errors)} of {len(val_analysis)} ({len(errors) / len(val_analysis):.1%})\n")
print("Most frequent confusions (true -> predicted):")
display(errors.groupby(["true", "predicted"]).size().sort_values(ascending=False).head(8).to_frame("count"))

print("High-confidence mistakes:")
display(errors.sort_values("confidence", ascending=False).head(10)[
    ["true", "predicted", "confidence", "medical_abstract"]])

errors.to_csv(f"{OUTPUT_DIR}/validation_errors.csv", index=False)

---
## 11. FINAL TEST EVALUATION — run once
⚠️ Run this section **only after** all decisions (model, epochs, learning rate) are final.
Do not change anything based on these numbers. The predictions file is what Person 5
uses for the master table and the final confusion matrices.

In [ ]:
test_preds, test_probs, test_time = predict(test_dataset)
y_test = test_df["condition_label"].values

test_results = evaluate_model(MODEL_LABEL, y_test, test_preds)
test_results.update({
    "Train Time (s)": round(TRAIN_TIME_SEC, 1),
    "Inference Time (s)": round(test_time, 2),
    "Checkpoint": MODEL_NAME,
})

print("TEST metrics:")
for metric, value in test_results.items():
    print(f"{metric}: {value:.6f}" if isinstance(value, float) else f"{metric}: {value}")
print("\nClassification report:")
print(classification_report(y_test, test_preds, labels=LABEL_IDS, target_names=LABEL_NAMES, digits=4))

pd.DataFrame([test_results]).to_csv(f"{OUTPUT_DIR}/{MODEL_LABEL.lower()}_test_results.csv", index=False)

# Shared prediction format: every model should export this same file.
test_predictions = pd.DataFrame({
    "abstract_id": test_df["abstract_id"],
    "true_label": y_test,
    "predicted_label": test_preds,
    "confidence": test_probs.max(axis=1).round(4),
})
for i, name in enumerate(LABEL_NAMES):
    test_predictions[f"prob_{name}"] = test_probs[:, i].round(4)
test_predictions.to_csv(f"{OUTPUT_DIR}/{MODEL_LABEL.lower()}_test_predictions.csv", index=False)
print("Saved test results and predictions.")

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, test_preds, labels=LABEL_IDS, display_labels=LABEL_NAMES,
    xticks_rotation=45, cmap="Blues"
)
plt.title(f"{MODEL_LABEL} - Test Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/test_confusion_matrix.png", dpi=150)
plt.show()

## 12. Download the outputs

In [ ]:
import shutil
from google.colab import files

shutil.rmtree(f"{OUTPUT_DIR}/checkpoints", ignore_errors=True)   # model weights are too large for GitHub
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))
files.download(f"{OUTPUT_DIR}.zip")